In [1]:
import duckdb

In [3]:
conn=duckdb.connect(r"D:\SEC.gov project\Analysis Project\credit_risk.db")

In [4]:
conn.sql("""
         SELECT COUNT(*)
         FROM num
         """).df()

,count_star()
0,2698851


In [6]:
conn.sql(""" 
         SUMMARIZE num
         """).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,adsh,VARCHAR,0000014195-19-000008,0001968915-25-000066,2724,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
1,tag,VARCHAR,AAMCommonStockIssuedperMPGCommonStockasPartoft...,warrantsWereExcluded,34157,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
2,version,VARCHAR,0000014195-19-000008,us-gaap/2025,2724,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
3,ddate,VARCHAR,02050930,20340331,255,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
4,qtrs,VARCHAR,0,96,76,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
5,uom,VARCHAR,AED,years,474,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
6,dimh,VARCHAR,0x00000000,0xfffb5f95df874cd57c334e0727324e5e,37138,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
7,iprx,VARCHAR,0,9,55,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,0.00
8,value,VARCHAR,-0.0001,999999.0000,272534,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,1.59
9,tag_key,VARCHAR,None,None,0,<NA>,<NA>,<NA>,<NA>,<NA>,2698851,100.00


In [7]:
conn.sql("""
         SELECT 
         adsh,
         tag,
         version,
         ddate,
         qtrs,
         uom,
         dimh,
         iprx,
         COUNT(*) as count
         FROM num 
         GROUP BY adsh, tag, version, ddate, qtrs, uom, dimh, iprx
         HAVING COUNT(*) > 1
         """).df()

,adsh,tag,version,ddate,qtrs,uom,dimh,iprx,count


No repeated primary key. 

In [11]:
conn.sql("""
    SELECT
        iprx,
        COUNT(*) AS count
    FROM num
    GROUP BY iprx
    ORDER BY iprx
""").df()

,iprx,count
0,0,2462484
1,1,193446
2,10,53
3,11,48
4,12,47
5,13,41
6,14,35
7,15,33
8,16,33
9,17,31


In [ ]:
conn.sql("""
    SELECT
        adsh,
        tag,
        version,
        ddate,
        qtrs,
        uom,
        dimh,
        COUNT(DISTINCT CAST(iprx AS INTEGER)) AS iprx_variants,
        MIN(CAST(iprx AS INTEGER)) AS min_iprx,
        MAX(CAST(iprx AS INTEGER)) AS max_iprx
    FROM num
    GROUP BY
        adsh,
        tag,
        version,
        ddate,
        qtrs,
        uom,
        dimh
    HAVING COUNT(DISTINCT CAST(iprx AS INTEGER)) > 1
    ORDER BY iprx_variants DESC, max_iprx DESC
""").df()

,adsh,tag,version,ddate,qtrs,uom,dimh,iprx_variants,min_iprx,max_iprx
0,0001113256-19-000096,DebtInstrumentInterestRateStatedPercentage,us-gaap/2019,20190930,0,pure,0x7cd733b0f32c087483fd911d320dd852,51,0,50
1,0000076605-25-000062,DebtInstrumentInterestRateStatedPercentage,us-gaap/2024,20211231,0,pure,0x53c400ab368cf05581b4fa847feb3dbd,44,0,43
2,0000076605-24-000078,DebtInstrumentInterestRateStatedPercentage,us-gaap/2023,20211231,0,pure,0x53c400ab368cf05581b4fa847feb3dbd,41,0,40
3,0000076605-23-000050,DebtInstrumentInterestRateStatedPercentage,us-gaap/2022,20180131,0,pure,0x194dbbe212fcb146acb01349c6fd95e7,41,0,40
4,0000076605-23-000050,DebtInstrumentInterestRateStatedPercentage,us-gaap/2022,20211231,0,pure,0x53c400ab368cf05581b4fa847feb3dbd,41,0,40
...,...,...,...,...,...,...,...,...,...,...
193441,0001819493-25-000128,EarningsPerShareDiluted,us-gaap/2025,20240630,2,USD,0x00000000,2,0,1
193442,0000355811-25-000041,IndefiniteLivedIntangibleAssetsExcludingGoodwill,us-gaap/2025,20241231,0,USD,0x5124d94d50f85e16686d521f4dbf81fd,2,0,1
193443,0000879526-22-000038,StockholdersEquity,us-gaap/2022,20220630,0,USD,0x2bea873ec009ac34c4d14deb5c0cc4f0,2,0,1
193444,0001477932-25-006059,NetIncomeLoss,us-gaap/2025,20250630,2,USD,0x00000000,2,0,1


In [17]:
conn.execute("""
             CREATE OR REPLACE TABLE clean.num AS 
             SELECT 
             adsh, 
             tag,
             version,
             strptime(ddate, '%Y%m%d')::DATE AS ddate,
             CAST(qtrs AS INTEGER) AS qtrs,
             uom,
             CAST(value AS DOUBLE) AS value
             FROM num
             WHERE iprx = 0 
             AND dimh = '0x00000000'
             """)

In [18]:
conn.close()

In [9]:
conn.sql("""SELECT 
    adsh, 
    tag, 
    version, 
    ddate, 
    qtrs, 
    uom, 
    COUNT(*) AS row_count,
    -- Group together the distinct values to see what the conflict is
    ARRAY_AGG(value) AS conflicting_values 
FROM num
WHERE dimh = '0x00000000' 
  AND iprx = 0
GROUP BY adsh, tag, version, ddate, qtrs, uom
HAVING COUNT(*) > 1;
""").df()

,adsh,tag,version,ddate,qtrs,uom,row_count,conflicting_values


Validation / exploratory query
* Purpose: Investigates whether multiple `iprx=0` records exist for the same otherwise-identical non-dimensional fact.
* Why: Validates the uniqueness assumptions used when reconstructing `clean.num`; unexpected duplicates should be investigated before applying Gold-layer calculations.
* Finding: No duplicate groups returned in this check.


In [14]:
conn.sql("SELECT count(*) FROM num WHERE dimh = '0x00000000' and iprx = 0").df()

,count_star()
0,1043305


Size check of the filtered data.

In [15]:
conn.sql("""SELECT 
    COUNT(DISTINCT adsh) as original_company_count,
    COUNT(DISTINCT CASE WHEN dimh = '0x00000000' AND iprx = 0 THEN adsh END) as filtered_company_count
FROM num;
""").df()

,original_company_count,filtered_company_count
0,2578,2578


Number of adsh filings are retained. Its' safe to filter.

In [16]:
conn.sql("""SELECT
    adsh,
    tag,
    version,
    ddate,
    qtrs,
    uom,
    dimh,
    ARRAY_AGG(value ORDER BY iprx) AS values_by_iprx
FROM num
WHERE dimh = '0x00000000'
  AND iprx IN (0, 1)
GROUP BY
    adsh,
    tag,
    version,
    ddate,
    qtrs,
    uom,
    dimh
HAVING COUNT(*) > 1
   AND COUNT(DISTINCT value) > 1;
""").df()

,adsh,tag,version,ddate,qtrs,uom,dimh,values_by_iprx
0,0001564590-21-024423,AccruedEnvironmentalLossContingenciesCurrent,us-gaap/2020,20210331,0,USD,0x00000000,"[1390000.0000, 1400000.0000]"
1,0001628280-23-014865,AccountsReceivableNetCurrent,us-gaap/2022,20230331,0,USD,0x00000000,"[171878000.0000, 171900000.0000]"
2,0001493152-23-010679,WarrantsAndRightsOutstanding,us-gaap/2022,20211130,0,USD,0x00000000,"[351240.0000, 7680156.0000]"
3,0001628280-21-015361,IncomeTaxExpenseBenefit,us-gaap/2021,20200630,2,USD,0x00000000,"[-10416000.0000, -10400000.0000]"
4,0001628280-21-015361,RestructuringCharges,us-gaap/2021,20200630,2,USD,0x00000000,"[3115000.0000, 3100000.0000]"
...,...,...,...,...,...,...,...,...
4001,0001290900-25-000010,AccountsReceivableNetCurrent,us-gaap/2025,20241231,0,USD,0x00000000,"[118683000.0000, 118700000.0000]"
4002,0000897077-20-000116,AmortizationOfIntangibleAssets,us-gaap/2020,20200630,1,USD,0x00000000,"[3613000.0000, 3600000.0000]"
4003,0000950170-25-002840,IncomeTaxExpenseBenefit,us-gaap/2024,20231130,1,USD,0x00000000,"[5976000.0000, 6000000.0000]"
4004,0001628280-25-024006,RetainedEarningsAccumulatedDeficit,us-gaap/2024,20250331,0,USD,0x00000000,"[-4324624000.0000, -4324600000.0000]"
